# H22a: Toy target-spectrum anisotropy sweep for Muon

**Counterpart identity.** This notebook is the explanatory counterpart to `run_experiment.py` in the same directory.
It deliberately **imports and runs the script's experiment function** instead of re-implementing the core training loop,
so the script remains the single source of truth for the toy study.

## Truthful scope of this first completion pass

This notebook preserves the current H22a study as a **toy sweep over target-spectrum anisotropy** in a 4-layer deep
linear regression problem.

- **What is directly controlled:** the target matrix power-law exponent $\alpha$, which changes target effective rank
  and target anisotropy.
- **What is only measured diagnostically:** the effective rank of the **initial gradients** induced by that setup.
- **What this notebook should *not* overclaim:** the current implementation does **not** yet provide a clean direct
  sweep from low to medium to high gradient effective rank.

The goal here is serious alignment and reporting: preserve the toy computation, expose raw results, add figures/tables,
and state clearly what is and is not demonstrated.

## Reproducibility, runtime, and notebook safety

This notebook avoids `__file__` and script-style path assumptions. It locates the experiment directory from the current
working directory, imports `run_experiment.py` by file path, and then uses the script's structured `run_experiment()`
API.

**Runtime note.** A full default run is not instantaneous because it performs many repeated training calls for LR sweeps.
The prior audit measured roughly **2m44s** for the full script on this machine. A small smoke mode is provided below for
quick notebook execution checks.

In [ ]:
from pathlib import Path
import importlib.util
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
plt.style.use('default')

RELATIVE_SCRIPT = Path('experiments/Tier_4_Falsification_Stress_Tests/H22a_ANISOTROPY_SWEET_SPOT/run_experiment.py')


def locate_h22a_script():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]

    for base in candidates:
        candidate = base / RELATIVE_SCRIPT
        if candidate.exists():
            return {
                'repo_root': base,
                'experiment_dir': candidate.parent,
                'script_path': candidate,
            }

    for base in candidates:
        candidate = base / 'run_experiment.py'
        if candidate.exists() and base.name == 'H22a_ANISOTROPY_SWEET_SPOT':
            repo_root = base.parents[2] if len(base.parents) >= 3 else base.parent
            return {
                'repo_root': repo_root,
                'experiment_dir': base,
                'script_path': candidate,
            }

    raise FileNotFoundError('Could not locate H22a run_experiment.py from the current working directory.')


paths = locate_h22a_script()
spec = importlib.util.spec_from_file_location('h22a_run_experiment', paths['script_path'])
h22a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h22a)

print('Repository root :', paths['repo_root'])
print('Experiment dir  :', paths['experiment_dir'])
print('Script path     :', paths['script_path'])

## Configuration and run mode

The script exposes defaults plus optional config overrides. By default, this notebook uses the script's full default
configuration. If the environment variable `H22A_NOTEBOOK_SMOKE=1` is set before execution, the notebook switches to a
small smoke configuration for faster verification.

In [ ]:
FULL_RUN_AUDIT_RUNTIME_SECONDS = 164.0
NOTEBOOK_SMOKE = os.environ.get('H22A_NOTEBOOK_SMOKE', '0') == '1'

if NOTEBOOK_SMOKE:
    CONFIG_OVERRIDES = {
        'num_steps': 25,
        'num_seeds': 3,
        'diagnostic_num_seeds': 2,
        'lr_sweep_num_seeds': 2,
        'alpha_values': [0.1, 1.0, 10.0],
        'lr_sgd': [1e-4, 1e-3, 1e-2],
        'lr_muon': [1e-4, 1e-3, 1e-2],
    }
else:
    CONFIG_OVERRIDES = None

default_config = h22a.get_default_config()
resolved_config = h22a.resolve_config(CONFIG_OVERRIDES)
seed_schedule = h22a.build_seed_schedule(resolved_config)
train_call_counts = h22a.count_train_calls(resolved_config)

print('Notebook smoke mode        :', NOTEBOOK_SMOKE)
print('Approx. audited full runtime:', f'{FULL_RUN_AUDIT_RUNTIME_SECONDS:.0f} s (~2m44s)')
print('Planned total train() calls :', train_call_counts['total_train_calls'])
print('  LR sweep train() calls    :', train_call_counts['lr_sweep_train_calls'])
print('  Final eval train() calls  :', train_call_counts['final_eval_train_calls'])
print('Seed schedule               :', seed_schedule)
print('\nResolved configuration:')
for key, value in resolved_config.items():
    print(f'  {key}: {value}')

## Run the script-backed experiment

This cell is the notebook's main execution entrypoint. It calls the script's `run_experiment()` function and stores the
returned structured raw results for the remaining analysis cells.

In [ ]:
run_started = time.time()
results = h22a.run_experiment(config_overrides=CONFIG_OVERRIDES, verbose=True)
notebook_wall_clock_seconds = time.time() - run_started

print('\nNotebook wall-clock runtime:', f'{notebook_wall_clock_seconds:.2f} s')
print('Script-reported runtime    :', f"{results['runtime']['elapsed_seconds']:.2f} s")
print('Interpretation note        :', results['summary']['interpretation_note'])

## Convert structured results into analysis tables

The script returns rich nested data: configuration, seeds, target diagnostics, measured initial-gradient diagnostics,
learning-rate sweep summaries, per-seed final losses, and the T1/T2 toy outcomes. The following cell turns those into
compact tables for inspection.

In [ ]:
alpha_rows = []
init_grad_seed_rows = []
final_seed_rows = []
lr_rows = []

for record in results['alpha_results']:
    alpha = record['alpha']
    target = record['target_diagnostics']
    init_grad = record['init_gradient_diagnostics']
    final_eval = record['final_eval']

    alpha_rows.append({
        'alpha': alpha,
        'target_eff_rank': target['target_effective_rank'],
        'target_eff_rank_frac': target['target_effective_rank_frac'],
        'sigma_max': target['sigma_max'],
        'sigma_min': target['sigma_min'],
        'anisotropy': target['anisotropy'],
        'init_grad_eff_rank': init_grad['mean_effective_rank'],
        'init_grad_eff_rank_std': init_grad['std_effective_rank'],
        'init_grad_eff_rank_frac': init_grad['mean_effective_rank_frac'],
        'init_grad_eff_rank_std_frac': (init_grad['std_effective_rank'] / resolved_config['dim']) if init_grad['std_effective_rank'] is not None else np.nan,
        'best_lr_sgd': record['best_lrs']['sgd'],
        'best_lr_muon': record['best_lrs']['muon'],
        'sgd_mean': final_eval['sgd_mean'],
        'sgd_std': final_eval['sgd_std'],
        'sgd_finite_count': final_eval['sgd_finite_count'],
        'muon_mean': final_eval['muon_mean'],
        'muon_std': final_eval['muon_std'],
        'muon_finite_count': final_eval['muon_finite_count'],
        'advantage_ratio': final_eval['advantage_ratio'],
        'log10_advantage': final_eval['log10_advantage'],
        'loss_gap': final_eval['loss_gap'],
    })

    for row in init_grad['per_seed']:
        init_grad_seed_rows.append({
            'alpha': alpha,
            'seed': row['seed'],
            'mean_layer_effective_rank': row['mean_layer_effective_rank'],
            'mean_layer_effective_rank_frac': row['mean_layer_effective_rank_frac'],
            'layer_effective_ranks': row['layer_effective_ranks'],
        })

    for row in final_eval['per_seed']:
        final_seed_rows.append({
            'alpha': alpha,
            'seed': row['seed'],
            'sgd_final_loss': row['sgd_final_loss'],
            'muon_final_loss': row['muon_final_loss'],
            'sgd_over_muon': row['sgd_over_muon'],
        })

    for optimizer in ['sgd', 'muon']:
        for lr_row in record['lr_sweep'][optimizer]['grid_results']:
            lr_rows.append({
                'alpha': alpha,
                'optimizer': optimizer,
                'lr': lr_row['lr'],
                'mean_loss': lr_row['mean_loss'],
                'std_loss': lr_row['std_loss'],
                'finite_count': lr_row['finite_count'],
                'losses': lr_row['losses'],
            })

summary_df = pd.DataFrame(alpha_rows).sort_values('alpha').reset_index(drop=True)
init_grad_seed_df = pd.DataFrame(init_grad_seed_rows).sort_values(['alpha', 'seed']).reset_index(drop=True)
final_seed_df = pd.DataFrame(final_seed_rows).sort_values(['alpha', 'seed']).reset_index(drop=True)
lr_df = pd.DataFrame(lr_rows).sort_values(['optimizer', 'alpha', 'lr']).reset_index(drop=True)

summary_display = summary_df.copy()
for column in ['target_eff_rank_frac', 'init_grad_eff_rank_frac', 'init_grad_eff_rank_std_frac']:
    summary_display[column] = 100.0 * summary_display[column]

print('Per-alpha summary table:')
print(summary_display.to_string(index=False, float_format=lambda x: f'{x:.4g}'))

## Target-spectrum diagnostics: what the experiment actually controls

The parameter $\alpha$ directly controls the target singular-value spectrum. That means target effective rank and target
anisotropy are the cleanly swept quantities in this implementation.

In [ ]:
target_table = summary_df[['alpha', 'sigma_max', 'sigma_min', 'anisotropy', 'target_eff_rank', 'target_eff_rank_frac']].copy()
target_table['target_eff_rank_frac'] = 100.0 * target_table['target_eff_rank_frac']
print(target_table.to_string(index=False, float_format=lambda x: f'{x:.4g}'))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(summary_df['alpha'], 100.0 * summary_df['target_eff_rank_frac'], marker='o', linewidth=2)
axes[0].set_xlabel('alpha (power-law exponent)')
axes[0].set_ylabel('Target effective-rank fraction (%)')
axes[0].set_title('Target effective rank vs alpha')
axes[0].grid(True, alpha=0.3)

axes[1].plot(summary_df['alpha'], summary_df['anisotropy'], marker='o', linewidth=2)
axes[1].set_yscale('log')
axes[1].set_xlabel('alpha (power-law exponent)')
axes[1].set_ylabel('Target anisotropy = sigma_max / sigma_min')
axes[1].set_title('Target anisotropy vs alpha')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Interpretation.** The target-spectrum sweep is strong: as $\alpha$ increases, target effective rank falls sharply and
anisotropy rises by many orders of magnitude. This part of the setup does what it is supposed to do.

## Measured initial gradient effective rank: what is only diagnosed

The original overclaim in H22a was to speak as though the current implementation already produced a broad direct sweep of
**gradient** effective rank. The script's own diagnostics show a more limited story: the measured initial gradient
spectrum changes only modestly across $\alpha$.

In [ ]:
grad_range = results['summary']['measured_init_gradient_effective_rank_frac_range']
target_range = results['summary']['target_effective_rank_frac_range']
grad_range_width_pp = 100.0 * (grad_range[1] - grad_range[0])

comparison_table = summary_df[['alpha', 'target_eff_rank_frac', 'init_grad_eff_rank_frac', 'init_grad_eff_rank_std_frac']].copy()
for column in ['target_eff_rank_frac', 'init_grad_eff_rank_frac', 'init_grad_eff_rank_std_frac']:
    comparison_table[column] = 100.0 * comparison_table[column]
print(comparison_table.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(summary_df['alpha'], 100.0 * summary_df['target_eff_rank_frac'], marker='s', linestyle='--', linewidth=2, label='Target effective-rank fraction')
ax.errorbar(
    summary_df['alpha'],
    100.0 * summary_df['init_grad_eff_rank_frac'],
    yerr=100.0 * summary_df['init_grad_eff_rank_std_frac'],
    marker='o',
    linewidth=2,
    capsize=4,
    label='Measured initial gradient effective-rank fraction',
)
ax.set_xlabel('alpha (power-law exponent)')
ax.set_ylabel('Effective-rank fraction (%)')
ax.set_title('Controlled target rank vs measured initial gradient rank')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Measured initial-gradient effective-rank fraction range: {100.0 * grad_range[0]:.1f}% to {100.0 * grad_range[1]:.1f}%")
print(f"Target effective-rank fraction range               : {100.0 * target_range[0]:.1f}% to {100.0 * target_range[1]:.1f}%")
print(f"Measured gradient-rank sweep width                 : {grad_range_width_pp:.1f} percentage points")

if grad_range_width_pp < 10.0:
    print('\nConclusion: in the current implementation, alpha does NOT meaningfully sweep initial gradient effective rank across distinct low/medium/high regimes.')
else:
    print('\nConclusion: the current implementation does produce a materially broad measured initial-gradient rank sweep.')

**Interpretation.** The notebook therefore needs to be careful: this is best described as a **target-spectrum anisotropy
sweep with gradient-rank diagnostics**, not as a settled demonstration of a true gradient effective-rank sweet spot.

## Final optimization outcomes

The script chooses learning rates by small LR sweeps, then evaluates SGD and Muon on the configured evaluation seeds.
The key reported metric is the ratio

\[
\text{advantage} = \frac{\text{mean final loss of SGD}}{\text{mean final loss of Muon}}.
\]

Because this can become very large when Muon reaches much smaller losses, the raw losses, chosen learning rates, and
finite-run counts are shown alongside the ratio.

In [ ]:
outcome_table = summary_df[[
    'alpha',
    'best_lr_sgd',
    'best_lr_muon',
    'sgd_mean',
    'sgd_std',
    'sgd_finite_count',
    'muon_mean',
    'muon_std',
    'muon_finite_count',
    'advantage_ratio',
    'log10_advantage',
]].copy()
print(outcome_table.to_string(index=False, float_format=lambda x: f'{x:.4g}'))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(summary_df['alpha'], summary_df['advantage_ratio'], marker='o', linewidth=2)
axes[0].set_xlabel('alpha (power-law exponent)')
axes[0].set_ylabel('Muon advantage = SGD mean loss / Muon mean loss')
axes[0].set_title('Final Muon advantage vs alpha')
axes[0].grid(True, alpha=0.3)

axes[1].plot(summary_df['alpha'], summary_df['sgd_mean'], marker='o', linewidth=2, label='SGD mean final loss')
axes[1].plot(summary_df['alpha'], summary_df['muon_mean'], marker='o', linewidth=2, label='Muon mean final loss')
axes[1].set_yscale('log')
axes[1].set_xlabel('alpha (power-law exponent)')
axes[1].set_ylabel('Mean final loss (log scale)')
axes[1].set_title('Raw optimizer losses vs alpha')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

### Per-seed final losses

The study is still small-sample by default, so it is useful to keep the per-seed final losses visible rather than only
showing aggregate means.

In [ ]:
print(final_seed_df.to_string(index=False, float_format=lambda x: f'{x:.4g}'))

## T1/T2 toy verdict and calibrated reading

The script preserves the original T1/T2 checks, but the interpretation now has to be calibrated by the gradient-rank
range actually achieved.

In [ ]:
tests_df = pd.DataFrame([
    {
        'test': name,
        'pass': info['pass'],
        'description': info['description'],
        'definition': info['definition'],
        'value': info['value'],
    }
    for name, info in results['tests'].items()
])
print(tests_df.to_string(index=False))

print('\nPeak alpha                               :', results['summary']['peak_alpha'])
print('Peak advantage ratio                     :', f"{results['summary']['peak_advantage_ratio']:.4g}x")
print('Peak measured init-grad eff-rank fraction:', f"{100.0 * results['summary']['peak_init_gradient_effective_rank_frac']:.1f}%")
print('Peak target eff-rank fraction            :', f"{100.0 * results['summary']['peak_target_effective_rank_frac']:.1f}%")
print('\nCalibrated note:')
print(results['summary']['interpretation_note'])

## Calibrated conclusion

This first completion pass leaves the toy study intact but makes the pair much more honest and reusable.

### What the current implementation now supports
- A reproducible **script-backed** sweep over target-spectrum anisotropy in a 4-layer deep linear task.
- Structured raw results suitable for downstream notebook analysis.
- Clear reporting of target diagnostics, measured initial-gradient diagnostics, chosen learning rates, per-seed final
  losses, optimizer means/stds, and T1/T2 outcomes.

### What the current implementation does **not** yet establish
- It does **not** establish a clean gradient effective-rank sweet spot.
- The reason is visible in the diagnostic plots above: target effective rank changes dramatically, but the measured
  initial gradient effective-rank fraction stays in a comparatively narrow band.

### Appropriate reading of H22a after this pass
Treat H22a as a **toy target-spectrum sweep with gradient-rank diagnostics**. Any stronger claim about a true gradient
rank sweet spot should wait for a later redesign in which the measured gradient effective rank itself is directly swept
across a broad low/medium/high range.